<!-- project-navigation -->
# 03_data_understanding_bureau_balance
Role: Exploration and data preparation.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [1]:
import pandas as pd
import os

# Chargement de la table bureau_balance
bureau_balance = pd.read_csv(
    "data/raw/bureau_balance.csv"
)

print("========== DIMENSION ==========")
print(bureau_balance.shape)


print("\n========== INFORMATIONS GENERALES ==========")
print(bureau_balance.info())


print("\n========== TYPES DES VARIABLES ==========")
print(bureau_balance.dtypes.value_counts())


print("\n========== VARIABLES ==========")
print(bureau_balance.columns.tolist())


print("\n========== VALEURS MANQUANTES ==========")
missing = bureau_balance.isna().sum()
missing_percent = (missing / len(bureau_balance)) * 100

missing_table = pd.DataFrame({
    "Nombre_manquant": missing,
    "Pourcentage": missing_percent
})

print(
    missing_table[missing_table["Nombre_manquant"] > 0]
    .sort_values("Pourcentage", ascending=False)
)


print("\n========== APERCU ==========")
display(bureau_balance.head())


print("\n========== ANALYSE STATUS ==========")
print(bureau_balance["STATUS"].value_counts(dropna=False))

========== DIMENSION ==========
(27299925, 3)

========== INFORMATIONS GENERALES ==========
<class 'pandas.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype
---  ------          -----
 0   SK_ID_BUREAU    int64
 1   MONTHS_BALANCE  int64
 2   STATUS          str  
dtypes: int64(2), str(1)
memory usage: 624.8 MB
None

========== TYPES DES VARIABLES ==========
int64    2
str      1
Name: count, dtype: int64

========== VARIABLES ==========
['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']

========== VALEURS MANQUANTES ==========
Empty DataFrame
Columns: [Nombre_manquant, Pourcentage]
Index: []

========== APERCU ==========


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C



========== ANALYSE STATUS ==========
STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64


In [2]:
import pandas as pd
import numpy as np


# ============================================================
# 1) Chargement des tables
# ============================================================

bureau_balance = pd.read_csv("data/raw/bureau_balance.csv")
bureau = pd.read_csv("data/raw/bureau.csv")


print("bureau_balance initial :", bureau_balance.shape)
print("bureau initial :", bureau.shape)



# ============================================================
# 2) Création des indicateurs comportementaux
# ============================================================

print("\nCréation des variables métier...")


# Nombre total de mois suivis pour chaque crédit
bb_features = bureau_balance.groupby("SK_ID_BUREAU").agg(

    # durée historique du crédit
    MONTHS_COUNT=("MONTHS_BALANCE", "count"),

    # mois minimum observé = ancienneté maximale
    MAX_HISTORY_MONTH=("MONTHS_BALANCE", "min"),

    # retard maximum observé
    MAX_DELAY_STATUS=("STATUS", 
                      lambda x: max(
                          [int(i) for i in x if i.isdigit()],
                          default=0
                      )),

    # nombre total de mois avec retard
    NB_DELAY_MONTHS=("STATUS",
                     lambda x: sum(
                         1 for i in x 
                         if i in ['1','2','3','4','5']
                     )),

    # nombre de mois clôturés
    NB_CLOSED_MONTHS=("STATUS",
                      lambda x: sum(
                          1 for i in x 
                          if i == "C"
                      )),

    # nombre de mois sans problème
    NB_OK_MONTHS=("STATUS",
                  lambda x: sum(
                      1 for i in x 
                      if i == "0"
                  ))
).reset_index()



# Taux de retard
bb_features["DELAY_RATIO"] = (
    bb_features["NB_DELAY_MONTHS"] /
    bb_features["MONTHS_COUNT"]
)


print("\n========== FEATURES PAR CREDIT ==========")
print(bb_features.head())

print("\nDimension après agrégation crédit :")
print(bb_features.shape)



# ============================================================
# 3) Fusion avec bureau
# ============================================================

print("\nFusion avec bureau...")


bureau_enriched = bureau.merge(
    bb_features,
    on="SK_ID_BUREAU",
    how="left"
)


print("Dimension après fusion :")
print(bureau_enriched.shape)



# ============================================================
# 4) Gestion des valeurs manquantes après fusion
# ============================================================

# Les crédits sans historique mensuel auront des NaN
# On considère qu'ils n'ont pas eu d'incident observé

cols_features = [
    "MONTHS_COUNT",
    "MAX_HISTORY_MONTH",
    "MAX_DELAY_STATUS",
    "NB_DELAY_MONTHS",
    "NB_CLOSED_MONTHS",
    "NB_OK_MONTHS",
    "DELAY_RATIO"
]


for col in cols_features:
    bureau_enriched[col] = bureau_enriched[col].fillna(0)



# ============================================================
# 5) Agrégation finale au niveau client
# ============================================================

print("\nAgrégation au niveau client...")


bureau_balance_features = bureau_enriched.groupby("SK_ID_CURR").agg(

    # nombre total de crédits
    NB_CREDITS_WITH_HISTORY=("SK_ID_BUREAU", "nunique"),

    # durée moyenne des historiques
    AVG_MONTHS_HISTORY=("MONTHS_COUNT", "mean"),

    # retard maximum rencontré
    MAX_DELAY=("MAX_DELAY_STATUS", "max"),

    # nombre total de mois en retard
    TOTAL_DELAY_MONTHS=("NB_DELAY_MONTHS", "sum"),

    # taux moyen de retard
    AVG_DELAY_RATIO=("DELAY_RATIO", "mean"),

    # nombre de crédits ayant eu un retard
    NB_CREDITS_WITH_DELAY=(
        "NB_DELAY_MONTHS",
        lambda x: (x > 0).sum()
    ),

    # mois sans problème
    TOTAL_OK_MONTHS=("NB_OK_MONTHS", "sum")

).reset_index()



# ============================================================
# 6) Vérification finale
# ============================================================

print("\n========== TABLE FINALE ==========")

print(bureau_balance_features.shape)

print("\nValeurs manquantes :")
print(bureau_balance_features.isna().sum().sum())


display(bureau_balance_features.head())


# ============================================================
# 7) Sauvegarde
# ============================================================

bureau_balance_features.to_csv(
    "data/processed/bureau_balance_features.csv",
    index=False
)


print("\nFichier sauvegardé : bureau_balance_features.csv")

bureau_balance initial : (27299925, 3)
bureau initial : (1716428, 17)

Création des variables métier...

========== FEATURES PAR CREDIT ==========
   SK_ID_BUREAU  MONTHS_COUNT  MAX_HISTORY_MONTH  MAX_DELAY_STATUS  \
0       5001709            97                -96                 0   
1       5001710            83                -82                 0   
2       5001711             4                 -3                 0   
3       5001712            19                -18                 0   
4       5001713            22                -21                 0   

   NB_DELAY_MONTHS  NB_CLOSED_MONTHS  NB_OK_MONTHS  DELAY_RATIO  
0                0                86             0          0.0  
1                0                48             5          0.0  
2                0                 0             3          0.0  
3                0                 9            10          0.0  
4                0                 0             0          0.0  

Dimension après agrégation crédit :

,SK_ID_CURR,NB_CREDITS_WITH_HISTORY,AVG_MONTHS_HISTORY,MAX_DELAY,TOTAL_DELAY_MONTHS,AVG_DELAY_RATIO,NB_CREDITS_WITH_DELAY,TOTAL_OK_MONTHS
0,100001,7,24.571429,1.0,1.0,0.007519,1,31.0
1,100002,8,13.750000,1.0,27.0,0.255682,6,45.0
2,100003,4,0.000000,0.0,0.0,0.000000,0,0.0
3,100004,2,0.000000,0.0,0.0,0.000000,0,0.0
4,100005,3,7.000000,0.0,0.0,0.000000,0,14.0



Fichier sauvegardé : bureau_balance_features.csv


# Traitement et transformation de la table `bureau_balance`

## 1. Objectif de la transformation

La table `bureau_balance` contient l'historique mensuel des crédits précédemment contractés par les clients.

Contrairement à la table `application_train` où :

- une ligne représente un client ;

la table `bureau_balance` contient :

- une ligne = un mois d'observation d'un crédit.

Un même crédit apparaît donc plusieurs fois avec différents états mensuels.

Exemple :

| SK_ID_BUREAU | MONTHS_BALANCE | STATUS |
|--------------|----------------|--------|
| 5715448 | 0 | C |
| 5715448 | -1 | C |
| 5715448 | -2 | 0 |
| 5715448 | -3 | 1 |

Le crédit `5715448` possède donc un historique mensuel permettant d'évaluer le comportement de remboursement du client.

L'objectif est de transformer ces observations mensuelles en variables synthétiques exploitables par un modèle de Machine Learning.

---

# 2. Analyse initiale de la table

## Dimension initiale

La table contient :

- 27 299 925 lignes ;
- 3 variables.

Chaque ligne correspond à une observation mensuelle d'un crédit.

Variables disponibles :

| Variable | Description |
|-|-|
| SK_ID_BUREAU | Identifiant du crédit historique |
| MONTHS_BALANCE | Mois relatif de suivi du crédit |
| STATUS | Etat du remboursement mensuel |

---

# 3. Analyse des valeurs manquantes

Une analyse des valeurs manquantes a été réalisée.

Résultat :

- aucune valeur manquante détectée.

Donc aucune méthode d'imputation n'a été nécessaire.

Cette étape est importante afin de vérifier la qualité des données avant transformation.

---

# 4. Analyse de la variable STATUS

La variable `STATUS` représente l'état mensuel du remboursement.

Signification :

| Valeur | Signification |
|-|-|
| C | Crédit clôturé |
| 0 | Aucun retard |
| X | Information inconnue |
| 1 | Retard de 1 à 30 jours |
| 2 | Retard de 31 à 60 jours |
| 3 | Retard de 61 à 90 jours |
| 4 | Retard de 91 à 120 jours |
| 5 | Retard supérieur à 120 jours |

Cette variable est importante pour l'analyse du risque car les retards historiques peuvent indiquer une probabilité plus élevée de défaut.

Cependant, elle n'est pas conservée directement sous cette forme catégorielle.

Elle est transformée en indicateurs comportementaux plus pertinents pour le modèle.

---

# 5. Première étape de transformation : niveau crédit

## Pourquoi une première agrégation ?

Un crédit possède plusieurs observations mensuelles.

Il faut donc résumer l'historique d'un crédit par une seule ligne.

La clé utilisée est :


SK_ID_BUREAU


La transformation :


bureau_balance
|
| Agrégation par SK_ID_BUREAU
↓
Features comportementales par crédit


---

# 6. Variables créées au niveau crédit

## Durée d'observation

Variable :


MONTHS_COUNT


Elle représente le nombre de mois disponibles pour observer le crédit.

Elle permet de mesurer la profondeur de l'historique.

---

## Ancienneté maximale

Variable :


MAX_HISTORY_MONTH


Elle correspond au mois le plus ancien observé.

Elle indique depuis combien de temps le crédit est suivi.

---

## Retard maximal observé

Variable :


MAX_DELAY_STATUS


Elle représente le niveau de retard maximum rencontré durant la vie du crédit.

Exemple :

Si un crédit possède les états :


0,0,1,2,0


alors :


MAX_DELAY_STATUS = 2


---

## Nombre de mois avec retard

Variable :


NB_DELAY_MONTHS


Elle compte le nombre de périodes où le client a eu un incident de paiement.

---

## Taux de retard

Variable :


DELAY_RATIO


Calcul :


DELAY_RATIO =
Nombre de mois avec retard /
Nombre total de mois observés


Cette variable permet de comparer les clients ayant des historiques de longueurs différentes.

---

# 7. Fusion intermédiaire avec la table `bureau`

## Pourquoi effectuer cette fusion ?

Cette fusion n'est pas la fusion finale avec `application_train`.

Elle permet uniquement d'enrichir chaque crédit avec son historique de paiement.

Les deux tables possèdent la même clé :


SK_ID_BUREAU


La relation est :


bureau_balance
|
| SK_ID_BUREAU
↓
bureau


Après fusion :

- la table `bureau` contient les caractéristiques financières du crédit ;
- les nouvelles variables contiennent son comportement de remboursement.

---

# 8. Deuxième étape : passage au niveau client

## Pourquoi une nouvelle agrégation ?

Dans le dataset final de Machine Learning, la prédiction concerne le client.

La variable cible :


TARGET


est définie au niveau :


SK_ID_CURR


Il faut donc obtenir :


1 ligne = 1 client


Transformation :


Crédits historiques
|
| Groupby SK_ID_CURR
↓
Features client


---

# 9. Variables finales créées

La table finale contient des indicateurs comme :

| Variable | Interprétation |
|-|-|
| NB_CREDITS_WITH_HISTORY | Nombre de crédits possédant un historique mensuel |
| AVG_MONTHS_HISTORY | Durée moyenne des historiques |
| MAX_DELAY | Retard maximal observé |
| TOTAL_DELAY_MONTHS | Nombre total de mois avec retard |
| AVG_DELAY_RATIO | Proportion moyenne de retard |
| NB_CREDITS_WITH_DELAY | Nombre de crédits ayant présenté un incident |
| TOTAL_OK_MONTHS | Nombre total de mois sans incident |

---

# 10. Résultat obtenu

Avant transformation :


bureau_balance

27 299 925 lignes
1 ligne = 1 mois d'un crédit


Après transformation :


bureau_balance_features

305 811 lignes environ
1 ligne = 1 client


Les variables obtenues représentent le comportement financier historique des clients.

---

# 11. Position dans le pipeline CRISP-DM

Cette étape appartient à :

## Phase 2 : Data Understanding

- compréhension de la structure des données ;
- analyse des variables ;
- identification de la signification métier.

## Phase 3 : Data Preparation

- création de nouvelles variables ;
- agrégation ;
- préparation des données pour le Machine Learning.

---

# 12. Prochaine étape

Les prochaines étapes sont :

1. Traiter la table `installments_payments`.
2. Créer les variables comportementales liées aux remboursements.
3. Réaliser la fusion finale :